# Fase 4 - Analisis exploratorio (Pharma Retail Sales USA, sintetico)

Corri el pipeline completo (generador -> unificar_datasets.py -> visualizar_resultados.py) y aqui respondo las preguntas de analisis que pide el mandato, usando `data/processed/dataset_unificado_mensual.csv`.

Recordatorio: esto es una muestra **sintetica**, no ventas reales de EE. UU.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/dataset_unificado_mensual.csv', parse_dates=['month'])
df['equiv_mes_completo'] = df['modeled_units_month'] / df['coverage_factor']
df.shape

(5828, 56)

## Pregunta 1: ¿que productos presentan mayor cambio porcentual respecto a 2019?

Comparo el promedio de `modeled_units_month` en 2019 vs 2020, usando solo meses completos (coverage_factor == 1) para que la comparacion sea justa.

In [2]:
medias = (df.loc[df['year'].isin([2019, 2020]) & (df['coverage_factor'] == 1)]
          .groupby(['product_id', 'product_name', 'sale_category', 'year'])['modeled_units_month']
          .mean().unstack('year'))
medias['cambio_pct'] = (medias[2020] / medias[2019] - 1) * 100

print('Los que mas bajaron:')
print(medias.sort_values('cambio_pct').head(5)[['cambio_pct']])
print()
print('Los que mas subieron:')
print(medias.sort_values('cambio_pct', ascending=False).head(5)[['cambio_pct']])

Los que mas bajaron:
year                                          cambio_pct
product_id product_name        sale_category            
RX-021     Doxycycline generic RX             -24.224244
RX-019     Amoxicillin generic RX             -21.841217
RX-023     Macrobid            RX             -21.813967
RX-020     Zithromax           RX             -21.696159
RX-022     Cefdinir generic    RX             -17.562461

Los que mas subieron:
year                                     cambio_pct
product_id product_name   sale_category            
OTC-002    Advil          OTC              5.627776
RX-005     Norvasc        RX               5.058297
RX-009     Eliquis        RX               4.835956
OTC-023    Afrin Original OTC              4.277568
RX-010     Xarelto        RX               4.007900


Los 5 que mas bajan son todos antibioticos RX (Doxycycline -24.2%, Amoxicillin -21.8%, Macrobid -21.8%, Zithromax -21.7%, Cefdinir -17.6%). Los que mas suben no superan el +6% y son una mezcla de OTC y RX sin un patron claro (Advil, Norvasc, Eliquis, Afrin, Xarelto). Esa asimetria (bajadas grandes y concentradas vs subidas chicas y dispersas) es la primera senal de que algo especifico se le metio a los antibioticos.

## Pregunta 2: ¿que grupos tienen trayectoria distinta durante la pandemia?

In [3]:
df['fase'] = pd.cut(
    df['month'],
    bins=[pd.Timestamp('2018-12-01'), pd.Timestamp('2020-03-01'), pd.Timestamp('2023-06-01'), pd.Timestamp('2026-11-01')],
    labels=['1_2019_prepandemia', '2_mar2020_may2023', '3_jun2023_oct2026'],
    right=False
)

resumen_fase = df.groupby(['fase', 'sale_category']).agg(
    media_producto_mes=('equiv_mes_completo', 'mean'),
    meses=('month', 'nunique')
).round(1)
resumen_fase

media_producto_mes  meses
fase               sale_category                           
1_2019_prepandemia CONTROLLED                 5805.8     14
                   OTC                        5844.0     14
                   RX                         5820.7     14
2_mar2020_may2023  CONTROLLED                 5225.6     39
                   OTC                        5842.8     39
                   RX                         5745.1     39
3_jun2023_oct2026  CONTROLLED                 4683.8     41
                   OTC                        5837.6     41
                   RX                         5843.9     41

Aqui se nota clarisimo: **CONTROLLED** (los 2 opioides) es el unico grupo que cae y no se recupera: pasa de ~5,806 (2019) a ~5,226 (pandemia) a ~4,684 (pospandemia, -19% vs 2019). **OTC** y **RX** se mantienen pratcticamente iguales en las 3 fases (entre 5,745 y 5,844). Nota: las 3 fases NO duran lo mismo (14, 39 y 41 meses), por eso comparo promedios por producto-mes y no sumas totales.

## Pregunta 3: ¿cuanto es calibracion por categoria y cuanto es ruido sintetico?

In [4]:
conteo_calibracion = df['calibration_level'].value_counts()
print(conteo_calibracion)
print()
print((conteo_calibracion / len(df) * 100).round(2), '%')

calibration_level
SYNTHETIC_NEUTRAL_NO_PRODUCT_EVIDENCE     5400
DOCUMENTED_CATEGORY_PROXY                  384
CARRY_FORWARD_CATEGORY_PROXY_SYNTHETIC      44
Name: count, dtype: int64

calibration_level
SYNTHETIC_NEUTRAL_NO_PRODUCT_EVIDENCE     92.66
DOCUMENTED_CATEGORY_PROXY                  6.59
CARRY_FORWARD_CATEGORY_PROXY_SYNTHETIC     0.75
Name: count, dtype: float64 %


El 92.7% de las filas (5,400 de 5,828) son `SYNTHETIC_NEUTRAL_NO_PRODUCT_EVIDENCE`, o sea multiplicador anual = 1 y solo varian por el ruido aleatorio del generador (`synthetic_monthly_product_noise_sd`). Solo el 6.6% (`DOCUMENTED_CATEGORY_PROXY`, 384 filas) trae un multiplicador basado en un proxy documentado (el indice de opioides del CDC para CONTROLLED, y un estudio de antibioticos para 5 productos RX). El resto chiquito (0.75%, `CARRY_FORWARD_CATEGORY_PROXY_SYNTHETIC`) son los meses 2025-2026 de opioides, donde se repite el ultimo dato documentado (2024) porque no hay data mas reciente.

O sea: la caida de CONTROLLED que se ve en la pregunta 2 **no es ruido**, es la calibracion documentada actuando sobre esos productos. Pero el resto de las variaciones mes a mes que se ven en el grafico (los "dientes de sierra") si son ruido sintetico, no reflejan nada real.

## Pregunta 4: ¿que observaciones tienen fuente documental y cuales son solo simulacion?

In [5]:
df.groupby('sale_category')['calibration_level'].value_counts()

sale_category  calibration_level                     
CONTROLLED     DOCUMENTED_CATEGORY_PROXY                  144
               CARRY_FORWARD_CATEGORY_PROXY_SYNTHETIC      44
OTC            SYNTHETIC_NEUTRAL_NO_PRODUCT_EVIDENCE     2820
RX             SYNTHETIC_NEUTRAL_NO_PRODUCT_EVIDENCE     2580
               DOCUMENTED_CATEGORY_PROXY                  240
Name: count, dtype: int64

Solo CONTROLLED (los 2 opioides) tiene observaciones con fuente documentada (CDC) en las 94 filas completas de esos 2 productos. Dentro de RX, 5 productos (los antibioticos: Doxycycline, Amoxicillin, Macrobid, Zithromax, Cefdinir) tambien tienen `DOCUMENTED_CATEGORY_PROXY` (el estudio de antibioticos). Los otros 55 productos (todo OTC + el resto de RX) son 100% `SYNTHETIC_NEUTRAL_NO_PRODUCT_EVIDENCE`: no tienen ninguna fuente detras, son pura simulacion.

## Pregunta 5: ¿como afecta el tamanio de muestra a `sample_records` sin cambiar el volumen modelado?

Corri el generador dos veces con la misma semilla (42) pero distinto `--base-records-per-month`.

In [6]:
comparacion_muestra = pd.DataFrame({
    'base_records_per_month': [200, 6200],
    'total_eventos_muestra': [18552, 575028],
    'total_unidades_modeladas': [33639168, 33639168],
})
comparacion_muestra

,base_records_per_month,total_eventos_muestra,total_unidades_modeladas
0,200,18552,33639168
1,6200,575028,33639168


Cambiar `--base-records-per-month` multiplica por 31 la cantidad de eventos de muestra (de 18,552 a 575,028), pero **el total de unidades modeladas no cambia ni un solo numero** (33,639,168 en ambos casos). Esto confirma lo que dice el mandato: ese parametro solo controla cuantos "eventos" sinteticos se generan para repartir el volumen ya fijado, no cambia la demanda modelada en si.

## Pregunta 6: ¿por que esto no es una medicion nacional ni evidencia causal?

- Los 62 productos arrancan con la **misma** referencia artificial (5,850 unidades/mes), asi que ningun ranking de "mas vendido" tiene sentido: la diferencia entre productos viene del generador, no del mercado real.
- El 92.7% de las observaciones no tiene ninguna fuente detras (`SYNTHETIC_NEUTRAL_NO_PRODUCT_EVIDENCE`).
- Los proxies de categoria (CDC, antibioticos) se aplican por igual a todos los productos del grupo, mes a mes, sin diferenciar entre ellos (`yearly_proxies_are_assigned_uniformly_to_months` = True en el manifest).
- No hay precios, pacientes, ni variables clinicas reales en el dataset.
- La muestra es "sintetica no probabilistica" (lo dice el propio manifest: `national_representativeness: False`, `market_estimate: False`).
- Una caida en 2020 en un dataset sintetico calibrado con un proxy de categoria no prueba que el COVID-19 causo esa caida en la realidad; solo muestra que el generador aplico ese supuesto.

Por todo esto, estos numeros sirven para practicar el flujo de ingenieria y analisis de datos (ETL, validacion, EDA), no para sacar conclusiones de mercado real.